# Link-level simulation

> Every bit of a transmission: codecs, modulation, noise, detection, and the channel built on them.

In [ ]:
#| default_exp link

In [ ]:
#| hide
from nbdev.showdoc import *

Link-level simulation in the sense of [Sionna PHY](https://nvlabs.github.io/sionna/phy/index.html): what happens
to the bits of one transmission.

1. The payload becomes bits (a `Codec`).
2. The bits become symbols (a `Modulation`).
3. The symbols go through the channel, `y = h √P x + n`, with `h` and `P` from the link budget (see `system`).
4. The receiver detects the bits and decodes them, so it gets the payload with its bit errors.

`LinkLevelChannel` puts these together as a comm-core `Channel`. It is slower than `SystemLevelChannel`, which
skips the bits, and it takes the same fading and power control: the same scenario runs at either level.

Each piece also works on its own and is vectorized: numpy arrays or torch tensors (on any device), with any batch
dimensions in front. Training code can therefore simulate a whole batch of links at once without a `Network` (see
the end of this page).

In [ ]:
#| export
from __future__ import annotations

import math
from abc import ABC, abstractmethod

import numpy as np

from comm_core.core import Message, Transmission, TransmissionResult
from comm_core.propagation import Fading
from comm_core.system import LinkBudget, PowerControl, RadioChannel, _is_torch
from comm_core.topology import Link

In [ ]:
#| export
def _arange(n: int, like):
    "0, ..., n-1: a torch tensor on `like`'s device if `like` is one, else a numpy array."
    if _is_torch(like):
        import torch
        return torch.arange(n, device=like.device)
    return np.arange(n)


def _conj(h):
    return h.conj() if _is_torch(h) else np.conj(h)


def _real(z):
    if _is_torch(z):
        return z.real if z.is_complex() else z
    return np.real(z)


def _as_array(x, dtype=None):
    "Tensors stay tensors; anything else becomes a numpy array (at least 1-d)."
    return x if _is_torch(x) else np.atleast_1d(np.asarray(x, dtype=dtype))

## Bits

Integers and their bits, least significant bit first. Both functions work on the last dimension.

In [ ]:
#| export
def to_bits(
    indices, # Integers, (..., M)
    bits_per_index: int, # Bits per integer
):
    "The bits of `indices`: (..., M * bits_per_index), least significant bit first."
    shifts = _arange(bits_per_index, indices)
    bits = (indices[..., None] >> shifts) & 1
    return bits.reshape(*indices.shape[:-1], -1)


def from_bits(
    bits, # 0/1 or booleans, (..., M * bits_per_index)
    bits_per_index: int, # Bits per integer
):
    "The integers of `bits` (the inverse of `to_bits`): (..., M)."
    weights = 2 ** _arange(bits_per_index, bits)
    return (bits.reshape(*bits.shape[:-1], -1, bits_per_index) * weights).sum(-1)

In [ ]:
x = np.array([[0, 1, 5, 255], [128, 7, 2, 3]])
b = to_bits(x, 8)
assert b.shape == (2, 32) and b[0, 8:16].tolist() == [1, 0, 0, 0, 0, 0, 0, 0]       # 1, least significant bit first
assert (from_bits(b, 8) == x).all() and (from_bits(b.astype(bool), 8) == x).all()

## Codecs

A `Codec` turns a payload into bits and back (source coding). `IndexCodec` is for discrete messages, e.g. indices
into a VQ codebook. `UniformQuantizer` is for real numbers, e.g. latents or positions.

In [ ]:
#| export
class Codec(ABC):
    "Payload <-> bits."

    @abstractmethod
    def encode(self, payload):
        "The payload's bits: (..., num_bits) of 0/1."

    @abstractmethod
    def decode(self, bits):
        "The payload of `bits`, which may have bit errors."


class IndexCodec(Codec):
    "Indices into a codebook (e.g. a VQ-VAE's), with ceil(log2(codebook_size)) bits each."

    def __init__(
        self,
        codebook_size: int = 256, # Number of codewords
    ):
        self.codebook_size = codebook_size
        self.bits_per_index = max(1, math.ceil(math.log2(codebook_size)))

    def encode(self, payload):
        "`payload`: integers in [0, codebook_size), (..., M)."
        x = _as_array(payload)
        if _is_torch(x):
            integer = not (x.is_floating_point() or x.is_complex()) and str(x.dtype) != 'torch.bool'
        else:
            integer = np.issubdtype(x.dtype, np.integer)
        if not integer:
            raise TypeError(f"IndexCodec encodes integers, not {x.dtype}")
        if (x < 0).any() or (x >= self.codebook_size).any():
            raise ValueError(f"indices must be in [0, {self.codebook_size})")
        return to_bits(x, self.bits_per_index)

    def decode(self, bits):
        "Indices (..., M). Bit errors can give indices >= codebook_size if it is not a power of 2."
        return from_bits(bits, self.bits_per_index)


class UniformQuantizer(Codec):
    "Real numbers in [low, high], each rounded to one of 2**bits evenly spaced levels (values outside are clipped)."

    def __init__(
        self,
        bits: int = 8, # Bits per number
        low: float = -1.0, # Smallest value
        high: float = 1.0, # Largest value
    ):
        self.bits, self.low, self.high = bits, low, high
        self.levels = 2 ** bits - 1

    def encode(self, payload):
        x = _as_array(payload, dtype=float)
        q = ((x.clip(self.low, self.high) - self.low) / (self.high - self.low) * self.levels).round()
        return to_bits(q.long() if _is_torch(q) else q.astype(np.int64), self.bits)

    def decode(self, bits):
        return self.low + from_bits(bits, self.bits) * ((self.high - self.low) / self.levels)

In [ ]:
c = IndexCodec(256)
assert c.bits_per_index == 8 and c.encode([3, 200]).shape == (16,) and c.decode(c.encode([3, 200])).tolist() == [3, 200]
assert IndexCodec(200).bits_per_index == 8 and IndexCodec(2).bits_per_index == 1
for bad, err in (([1.5], TypeError), ([True], TypeError), ([256], ValueError), ([-1], ValueError)):
    try:
        c.encode(bad); raise AssertionError(bad)
    except err:
        pass
q = UniformQuantizer(bits=8, low=-1, high=1)
x = np.array([-1.0, -0.3, 0.0, 0.77, 1.0, 3.0])
y = q.decode(q.encode(x))
assert y.shape == x.shape and np.abs(y - x.clip(-1, 1)).max() <= 1 / 255 + 1e-12      # within half a step (2/255 / 2)

## Modulation

A `Modulation` maps bits to symbols with average energy 1. It detects them again knowing the channel coefficient
`h` (coherent detection, with perfect channel knowledge at the receiver).

In [ ]:
#| export
class Modulation(ABC):
    "Bits <-> symbols."

    bits_per_symbol: int = 1

    @abstractmethod
    def modulate(self, bits):
        "Symbols (..., num_bits / bits_per_symbol), with average energy 1."

    @abstractmethod
    def demodulate(self, received, h):
        "The detected bits (booleans) of the received symbols, knowing the channel coefficients `h`."


class BPSK(Modulation):
    "Binary phase-shift keying: bit 0 -> -1, bit 1 -> +1."

    bits_per_symbol = 1

    def modulate(self, bits):
        return 2.0 * bits - 1.0

    def demodulate(self, received, h):
        "Matched filter, which is the maximum-likelihood detector for BPSK: the sign of Re(y h*)."
        return _real(received * _conj(h)) > 0

In [ ]:
show_doc(BPSK.demodulate)

In [ ]:
bpsk = BPSK()
b = np.array([0, 1, 1, 0])
assert bpsk.modulate(b).tolist() == [-1, 1, 1, -1]
h = np.exp(1j * 2.0)                                               # any phase: the receiver knows h
assert (bpsk.demodulate(h * bpsk.modulate(b), h) == b).all()

## Noise

`propagate` is what the receiver gets: `y = h √P x + n`, with complex Gaussian noise `n ~ CN(0, noise_power)`.
`h` and `P` broadcast against the symbols. With one value per packet, i.e. shape `(..., 1)`, the channel is flat and
constant during the packet.

In [ ]:
#| export
def awgn(
    like, # Array or tensor whose shape (and device) the noise takes
    noise_power, # Noise power (variance of the complex noise)
    rng=None, # numpy: a Generator or seed; torch: a torch.Generator or None
):
    "Complex Gaussian noise, CN(0, noise_power)."
    std = (noise_power / 2) ** 0.5
    if _is_torch(like):
        import torch
        re = torch.randn(like.shape, generator=rng, device=like.device)
        im = torch.randn(like.shape, generator=rng, device=like.device)
        return torch.complex(re, im) * std
    rng = np.random.default_rng(rng)
    shape = np.shape(like)
    return (rng.standard_normal(shape) + 1j * rng.standard_normal(shape)) * std


def propagate(
    symbols, # Transmitted symbols x, (..., num_symbols)
    h, # Channel coefficients (complex), broadcast against the symbols, e.g. (..., 1): one per packet
    power, # Transmit power P, broadcast against the symbols
    noise_power, # Noise power at the receiver
    rng=None, # Random generator of the noise (see `awgn`)
):
    "What the receiver gets: y = h √P x + n."
    signal = h * power ** 0.5 * symbols
    return signal + awgn(signal, noise_power, rng)

In [ ]:
from comm_core.system import bpsk_bit_error_rate, packet_error_rate

In [ ]:
# the simulated bit error rate of BPSK is the analytic one (the PHY abstraction of `system`)
rng = np.random.default_rng(0)
b = rng.integers(0, 2, 200_000)
for snr_db in (0, 4, 7):
    noise_power = 10 ** (-snr_db / 10)                       # P = 1, |h| = 1
    h = np.exp(1j * 0.7)
    errors = (bpsk.demodulate(propagate(bpsk.modulate(b), h, 1.0, noise_power, rng), h) != b).mean()
    expected = bpsk_bit_error_rate(10 ** (snr_db / 10))
    assert abs(errors - expected) < 0.1 * expected + 2e-4, (snr_db, errors, expected)
n = awgn(np.zeros(100_000), 2.0, rng)
assert n.dtype == complex and abs((np.abs(n) ** 2).mean() - 2.0) < 0.05

## Link-level channel

`LinkLevelChannel` simulates every bit of every transmission, using the pieces above and the link budget of
`RadioChannel` (see `system`).

* **Delivery.** A sent message is delivered with its bit errors (`success=True`): the payload the receiver gets may
  differ from what was sent.
* **No CRC by default.** Pass `drop_corrupted=True` to drop corrupted packets instead, like a receiver that checks a
  CRC.
* **Outages** are lost, as at the system level.

`size_bits` is what the codec produced. `error_rate` is the transmission's bit error rate, and `metadata` adds
`bit_errors`.

In [ ]:
#| export
class LinkLevelChannel(RadioChannel):
    "Every bit simulated: payload -> bits -> symbols -> h √P x + n -> detected bits -> payload (with its bit errors)."

    def __init__(
        self,
        codec: Codec | None = None, # Payload <-> bits; None: `IndexCodec(256)`
        modulation: Modulation | None = None, # Bits <-> symbols; None: `BPSK()`
        fading: Fading | None = None, # The coefficient h of a transmission; None: h = 1
        power_control: PowerControl | None = None, # Transmit power and outages; None: the link's tx_power_w
        noise_power: float = 1.0, # Noise power at the receiver (same unit as the power)
        drop_corrupted: bool = False, # Lose the packets with bit errors (a CRC) instead of delivering them
        seed: int | None = None, # Seed of the noise and the fading
    ):
        self.codec = codec if codec is not None else IndexCodec()
        self.modulation = modulation if modulation is not None else BPSK()
        self.drop_corrupted = drop_corrupted
        super().__init__(fading, power_control, noise_power, seed)

    def transmit(
        self,
        transmission: Transmission,
        link: Link,
        interference: float = 0.0, # Power of the other transmissions at the receiver (see `InterferenceBackend`)
        budget: LinkBudget | None = None, # The transmission's link budget, if already computed
    ) -> TransmissionResult:

        if budget is None:
            budget = self.link_budget(transmission, link)
        if not budget.transmit:
            return self._outage(transmission, budget)
        sinr_db = self._sinr_db(budget, interference)
        extra = {'interference_power': interference} if interference else {}
        if sinr_db == -math.inf:                         # drowned (e.g. an interferer at the receiver's place)
            return self._lost(transmission, link, budget, sinr_db, **extra)
        message = transmission.message
        bits = np.asarray(self.codec.encode(message.payload))
        num_bits = int(bits.shape[-1])
        if budget.snr_db == math.inf:                    # same place: a perfect link
            return TransmissionResult(success=True, transmission=transmission, payload=message.payload,
                                      size_bits=int(bits.size), snr_db=math.inf, error_rate=0.0,
                                      metadata=self._metadata(budget, bit_errors=0))

        k = self.modulation.bits_per_symbol
        padded = np.pad(bits, [(0, 0)] * (bits.ndim - 1) + [(0, -num_bits % k)])
        symbols = self.modulation.modulate(padded)
        received = propagate(symbols, budget.h, budget.power, self.noise_power + interference, self.rng)  # interference as noise
        detected = np.asarray(self.modulation.demodulate(received, budget.h))[..., :num_bits]
        bit_errors = int((detected != bits).sum())
        success = not (self.drop_corrupted and bit_errors)
        latency, energy = self._timing(budget, int(np.size(symbols)), link)
        return TransmissionResult(
            success=success,
            transmission=transmission,
            payload=self.codec.decode(detected) if success else None,
            latency=latency,
            size_bits=int(bits.size),
            energy=energy,
            snr_db=sinr_db,
            error_rate=bit_errors / max(int(bits.size), 1),
            metadata=self._metadata(budget, bit_errors=bit_errors, **extra),
        )

    def _num_bits(self, message: Message) -> int:
        return int(np.size(self.codec.encode(message.payload)))

    def _num_symbols(self, num_bits: int) -> int:
        return -(-num_bits // self.modulation.bits_per_symbol)

### Tests

In [ ]:
from comm_core.core import Message
from comm_core.propagation import NoFading, RayleighFading
from comm_core.system import ChannelInversion, MessagePower, SystemLevelChannel

In [ ]:
def send(ch, payload, link=Link('a', 'b'), n=1, **metadata):
    t = Transmission(message=Message('a', 'b', payload, metadata=metadata), sender='a', receiver='b', start_time=0.0)
    return [ch.transmit(t, link) for _ in range(n)]

idx = np.arange(49) * 5                                          # 49 codebook indices
r, = send(LinkLevelChannel(noise_power=1e-6, seed=0), idx)        # 60 dB: no errors
assert r.success and (r.payload == idx).all() and r.size_bits == 49 * 8 and r.metadata['bit_errors'] == 0
assert abs(r.snr_db - 60) < 1e-9 and r.energy == 49 * 8 and r.latency == 0
# 0 dB: about 8% of the bits are wrong, and the receiver gets the corrupted indices
rs = send(LinkLevelChannel(noise_power=1.0, seed=0), idx, n=50)
ber = np.mean([r.error_rate for r in rs])
assert all(r.success for r in rs) and abs(ber - bpsk_bit_error_rate(1.0)) < 0.01
assert not (rs[0].payload == idx).all()
# a CRC: corrupted packets are lost
assert not any(r.success for r in send(LinkLevelChannel(noise_power=1.0, drop_corrupted=True, seed=0), idx, n=20))
# channel inversion: the weak link (|h|^2 = 0.01) would need 1000 > 10: outage, nothing sent
for h, sent in ((1.2, True), (0.1, False)):
    r, = send(LinkLevelChannel(fading=NoFading(h), power_control=ChannelInversion(10, max_power=10), seed=0), idx)
    assert r.success == sent and r.size_bits == (392 if sent else 0) and r.metadata.get('outage', False) != sent
    if sent:
        assert abs(r.snr_db - 10) < 1e-6
# the sender chooses the power
ch = LinkLevelChannel(power_control=MessagePower(), seed=0)
assert not send(ch, idx)[0].success and send(ch, idx, tx_power=100.0)[0].metadata['tx_power'] == 100.0
# a finite bandwidth: latency = symbols / bandwidth, energy = power x latency
r, = send(LinkLevelChannel(noise_power=1e-6), idx, link=Link('a', 'b', bandwidth_hz=1e6, tx_power_w=0.5))
assert abs(r.latency - 392e-6) < 1e-15 and abs(r.energy - 0.5 * 392e-6) < 1e-15
# Rayleigh fading: varying SNR, reproducible with reset(seed)
ch = LinkLevelChannel(fading=RayleighFading(), noise_power=0.1)
ch.reset(seed=1); first = [r.snr_db for r in send(ch, idx, n=5)]
ch.reset(seed=1); assert [r.snr_db for r in send(ch, idx, n=5)] == first and len(set(first)) == 5
# real numbers: quantized to 8 bits
r, = send(LinkLevelChannel(codec=UniformQuantizer(8, -1, 1), noise_power=1e-6), np.array([0.25, -0.5]))
assert r.size_bits == 16 and np.allclose(r.payload, [0.25, -0.5], atol=1 / 255)

The two levels agree. With a CRC, the link-level channel loses packets as often as the system level's PHY abstraction
says it does:

In [ ]:
idx = np.arange(8) * 30                                          # 64 bits
for snr_db in (4, 6, 8):
    noise = 10 ** (-snr_db / 10)
    link_level = LinkLevelChannel(noise_power=noise, drop_corrupted=True, seed=0)
    lost = 1 - np.mean([r.success for r in send(link_level, idx, n=1000)])
    expected = packet_error_rate(snr_db, 64)
    assert abs(lost - expected) < 0.04, (snr_db, lost, expected)

With interference (see `InterferenceBackend`), the link level adds it to the noise of every bit, and gives the BER of the SINR:

In [ ]:
from comm_core.propagation import RayTracedFading
from comm_core.system import InterferenceBackend

In [ ]:
xs = np.arange(5)
d = np.abs(np.subtract.outer(xs, xs)).astype(float)
table = RayTracedFading(np.where(d > 0, 1e-3 / np.maximum(d, 1) ** 1.5, 0), [(x, 0) for x in xs],
                        node_positions={'a': (0, 0), 'b': (1, 0), 'c': (2, 0), 'd': (3, 0)})
backend = InterferenceBackend(LinkLevelChannel(fading=table, noise_power=1e-10, seed=0))
pair = lambda s, r: (Transmission(Message(s, r, np.arange(256) % 256), s, r, 0.0), Link(s, r))
r, _ = backend.transmit_all([pair('a', 'b'), pair('c', 'd')])        # c is as close to b as a: SINR about 0 dB
sinr = 1e-6 / (1e-10 + 1e-6)
assert abs(r.snr_db - 10 * np.log10(sinr)) < 1e-9 and abs(r.metadata['interference_power'] - 1e-6) < 1e-18
assert abs(r.error_rate - bpsk_bit_error_rate(sinr)) < 0.02 and r.success

## Batched, without a network

Training code can apply the same steps to a whole batch of links at once, on numpy arrays or torch tensors, e.g.
`B·T` samples × `n` neighbours, each sending 49 codebook indices, with channel inversion. Here `csi` holds the
coefficients `(B·T, n, 1)`, and links in an outage get the null index `codebook_size`:

```python
codec, bpsk = IndexCodec(256), BPSK()
power, send = ChannelInversion(target_snr_db=10, max_power=10).allocate(csi.abs() ** 2, noise_power)
x = bpsk.modulate(codec.encode(msg_indices[:, None, :]))           # (B·T, 1, 392), the same message to every neighbour
y = propagate(x, csi, power, noise_power)                          # (B·T, n, 392)
received = codec.decode(bpsk.demodulate(y, csi))                   # (B·T, n, 49)
received = torch.where(send, received, codec.codebook_size)        # nothing received in an outage
```

In [ ]:
#| hide
def batch(xp, msg_indices, csi, noise_power=1.0, rng=None):
    codec, bpsk = IndexCodec(256), BPSK()
    power, send = ChannelInversion(target_snr_db=10, max_power=10).allocate(abs(csi) ** 2, noise_power)
    x = bpsk.modulate(codec.encode(msg_indices[:, None, :]))
    y = propagate(x, csi, power, noise_power, rng)
    received = codec.decode(bpsk.demodulate(y, csi))
    return xp.where(send, received, codec.codebook_size), send

rng = np.random.default_rng(0)
msg = rng.integers(0, 256, (64, 49))
csi = (rng.standard_normal((64, 3, 1)) + 1j * rng.standard_normal((64, 3, 1))) / np.sqrt(2)
received, sent = batch(np, msg, csi, rng=rng)
assert received.shape == (64, 3, 49) and ((received == 256) == ~sent).all()
ber = (to_bits(np.where(sent, received, 0), 8) != to_bits(np.broadcast_to(msg[:, None], received.shape), 8))[sent[..., 0]].mean()
assert abs(ber - bpsk_bit_error_rate(10.0)) < 1e-3                # every sent link at 10 dB
try:
    import torch
except ImportError:
    torch = None
if torch is not None:                                               # the same on torch tensors
    g = torch.Generator().manual_seed(0)
    received_t, sent_t = batch(torch, torch.as_tensor(msg), torch.as_tensor(csi), rng=g)
    assert received_t.shape == (64, 3, 49) and torch.equal(sent_t, torch.as_tensor(sent))
    assert ((received_t == 256) == ~sent_t).all()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()